# Stemming VS Lemmatization in Sentiment Analysis of Movies reviews

## Import and Load data

In [1]:
import nltk
import random
import pandas as pd
import re
from nltk.corpus import movie_reviews,wordnet,stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer,WordNetLemmatizer
from nltk import pos_tag

In [2]:
nltk.download('movie_reviews')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\asus\AppData\Roaming\nl

True

In [3]:
movie_reviews

<CategorizedPlaintextCorpusReader in 'C:\\Users\\asus\\AppData\\Roaming\\nltk_data\\corpora\\movie_reviews'>

In [4]:
docs = [(movie_reviews.raw(fileid),category)
        for category in movie_reviews.categories()
        for fileid in movie_reviews.fileids(category)]

In [5]:
random.shuffle(docs)
docs = docs[:200]

df = pd.DataFrame(docs,columns=['reviews','label'])
df.label =df.label.map({'pos':1,'neg':0})

In [6]:
df.head()

,reviews,label
0,playwright tom stoppard and screenwriter marc ...,1
1,michael crichton has had a long career of writ...,0
2,capsule : the world will come to an end at mid...,1
3,we share the descent into darkness of a talent...,1
4,"originally launched in 1978 , this popular fil...",1


In [7]:
df.tail()

,reviews,label
195,"edward zwick's "" the siege "" raises more quest...",1
196,after a marketing windup of striking visuals a...,0
197,"synopsis : upper middle class , suburban famil...",1
198,"the premise of this movie is , well , pretty f...",0
199,"deep rising is one of "" those "" movies . \nthe...",1


In [8]:
df.columns

Index(['reviews', 'label'], dtype='str')

In [9]:
df.label.value_counts()

label
0    103
1     97
Name: count, dtype: int64

## Text Cleaning

In [10]:
def clean_text(text):
    text= text.lower()
    text =re.sub(r"[^\w\s]","",text)
    return text
df['clean'] = df.reviews.apply(clean_text)

In [11]:
df.clean.head()

0    playwright tom stoppard and screenwriter marc ...
1    michael crichton has had a long career of writ...
2    capsule  the world will come to an end at midn...
3    we share the descent into darkness of a talent...
4    originally launched in 1978  this popular film...
Name: clean, dtype: str

## Preprocessing Functions

In [12]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [13]:
stop_words = set(stopwords.words('english')) - {'not','no','never'}
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'nor',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'only',
 'or',
 'o

In [14]:
# Stemming

stemmer = PorterStemmer()
def stem_pipeline(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    stemmed = [stemmer.stem(w) for w in filtered]
    return " ".join(stemmed)

In [15]:
df['stemmed'] = df.clean.apply(stem_pipeline)

In [16]:
df.stemmed.head()

0    playwright tom stoppard screenwrit marc norman...
1    michael crichton long career write novel mani ...
2    capsul world come end midnight everyon know mu...
3    share descent dark talent boy pianist year lat...
4    origin launch 1978 popular film reintroduc 199...
Name: stemmed, dtype: str

## Lemmatizaton

In [17]:
lemmztizer = WordNetLemmatizer()
def lemma_pipeline(text):
    tokens =word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    pos_tags = pos_tag(filtered)
    lemmztized = [lemmztizer.lemmatize(w,get_wordnet_pos(tag)) for w, tag in pos_tags]
    return " ".join(lemmztized)

In [18]:
df['lemmatized'] = df.clean.apply(lemma_pipeline)

In [19]:
df.lemmatized.head()

0    playwright tom stoppard screenwriter marc norm...
1    michael crichton long career write novel many ...
2    capsule world come end midnight everyone know ...
3    share descent darkness talented boy pianist ye...
4    originally launch 1978 popular film reintroduc...
Name: lemmatized, dtype: str

# Classification and Accuracy

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [21]:
x_stem =df.stemmed
x_lemma = df.lemmatized
y = df.label

In [22]:
x_stem.head()

0    playwright tom stoppard screenwrit marc norman...
1    michael crichton long career write novel mani ...
2    capsul world come end midnight everyon know mu...
3    share descent dark talent boy pianist year lat...
4    origin launch 1978 popular film reintroduc 199...
Name: stemmed, dtype: str

In [23]:
x_lemma.head()

0    playwright tom stoppard screenwriter marc norm...
1    michael crichton long career write novel many ...
2    capsule world come end midnight everyone know ...
3    share descent darkness talented boy pianist ye...
4    originally launch 1978 popular film reintroduc...
Name: lemmatized, dtype: str

In [24]:
y.head()

0    1
1    0
2    1
3    1
4    1
Name: label, dtype: int64

In [25]:
x_train_s,x_test_s,y_train , y_test = train_test_split(x_stem,y,test_size=0.3,random_state=42)

In [26]:
x_train_l,x_test_l,_,_ = train_test_split(x_lemma,y,test_size=0.3,random_state=42)

In [27]:
x_train_s.shape

(140,)

In [35]:
y_train.shape

(140,)

In [36]:
# TF-IDF Vectorization

vectorizer  = TfidfVectorizer()
x_train_s_vec = vectorizer.fit_transform(x_train_s)
x_test_s_vec = vectorizer.transform(x_test_s)
x_train_s_vec

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 34477 stored elements and shape (140, 8618)>

In [37]:
x_train_l_vec = vectorizer.fit_transform(x_train_l)
x_test_l_vec = vectorizer.transform(x_test_l)

In [38]:
# Train and Evaluate 
model_s = LogisticRegression(max_iter=1000)
model_s.fit(x_train_s_vec,y_train)
y_predict_s = model_s.predict(x_test_s_vec)
acc_s = accuracy_score(y_test,y_predict_s)
print("Stemming Accuracy:",round(acc_s,2))

Stemming Accuracy: 0.62


In [40]:
model_l = LogisticRegression(max_iter=1000)
model_l.fit(x_train_l_vec,y_train)
y_predict_l = model_l.predict(x_test_l_vec)
acc_l = accuracy_score(y_test,y_predict_l)
print("Lemmatization Accuracy:",round(acc_s,2))

Lemmatization Accuracy: 0.62
